# Task 1 - Financial AI Equity Research
Run live first; use `allow_demo_fallback=True` only for an explicitly labelled offline run.

In [ ]:
from pathlib import Path
import sys
task_root = Path.cwd().parent if (Path.cwd().parent / 'src').exists() else Path.cwd() / 'task1_financial'
sys.path.insert(0, str(task_root.parent))
from task1_financial.src.pipeline import YFinanceMarketAdapter, add_indicators, build_snapshot
from task1_financial.src.reasoning import analyze_news_and_signal
from task1_financial.src.render import render_markdown, write_html
import asyncio, json
adapter = YFinanceMarketAdapter()
market = adapter.fetch('AAPL', years=2, allow_demo_fallback=False)
snapshot = build_snapshot(market)
research = asyncio.run(analyze_news_and_signal(snapshot))
research.report_markdown = render_markdown(research)
out = task_root / 'artifacts' / 'equity_brief.md'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(research.report_markdown, encoding='utf-8')
write_html(research, out.with_suffix('.html'))
if not market.bars.empty:
    analytical = market.bars.copy()
    analytical['raw_close'] = market.bars['close']
    if 'adj_close' in analytical.columns and analytical['adj_close'].notna().any():
        analytical['close'] = analytical['adj_close'].where(analytical['adj_close'].notna(), analytical['close'])
    add_indicators(analytical).to_csv(out.with_name('AAPL_market.csv'), index_label='timestamp')
    out.with_name('AAPL_metadata.json').write_text(json.dumps({'provenance': snapshot.provenance.model_dump(mode='json'), 'limitations': snapshot.limitations}, indent=2), encoding='utf-8')
print(research.model_dump_json(indent=2))